## Evaluate Finetuned model

the CV folds are use to pick a threshold. eval_finetune.py on the CV folds with a set of thresholds computes mDice per threshold. The best threshold found there shall then be used to evaluate the retrained (on the complete dataset) 'all' model.

In [8]:
import re
import pandas as pd

### first threshold sweep

in a first round i used a broad sweep to get a baseline for a good threshold:

In [11]:
path = '/home/justs/detection_segmentation/IEE/vein_segmentation/slurm/out/eval_finetune/eval_finetune_scan_and_photo_CV_11107122.out'

rows = []
with open(path) as f:
    for line in f:
        m = re.match(r'fold(\d+) (pretrained|fine-tuned)\s+(.*)', line)
        if m:
            fold, model, rest = m.groups()
            for t, s in re.findall(r'(\d\.\d+): (\d\.\d+)', rest):
                rows.append((int(fold), model, float(t), float(s)))

df = pd.DataFrame(rows, columns=['fold', 'model', 'threshold', 'mDice'])   # long format

mean = df.pivot_table(index='threshold', columns='model', values='mDice')  # mean over folds
mean['gain'] = mean['fine-tuned'] - mean['pretrained']
print(mean.round(4))

model      fine-tuned  pretrained    gain
threshold                                
0.30           0.0077      0.0077  0.0000
0.40           0.3939      0.0077  0.3862
0.50           0.4854      0.4263  0.0591
0.60           0.5270      0.4834  0.0436
0.70           0.5622      0.5491  0.0131
0.80           0.5986      0.6005 -0.0018
0.90           0.6350      0.6069  0.0281
0.95           0.6125      0.5640  0.0485


=> best model sits at the edge of the sweep, so another finer sweep is done to find the best threshold

In [12]:
path = '/home/justs/detection_segmentation/IEE/vein_segmentation/slurm/out/eval_finetune/eval_finetune_scan_and_photo_CV_11107201.out'

rows = []
with open(path) as f:
    for line in f:
        m = re.match(r'fold(\d+) (pretrained|fine-tuned)\s+(.*)', line)
        if m:
            fold, model, rest = m.groups()
            for t, s in re.findall(r'(\d\.\d+): (\d\.\d+)', rest):
                rows.append((int(fold), model, float(t), float(s)))

df = pd.DataFrame(rows, columns=['fold', 'model', 'threshold', 'mDice'])   # long format

mean = df.pivot_table(index='threshold', columns='model', values='mDice')  # mean over folds
mean['gain'] = mean['fine-tuned'] - mean['pretrained']
print(mean.round(4))

model      fine-tuned  pretrained    gain
threshold                                
0.85           0.6178      0.6146  0.0032
0.86           0.6219      0.6144  0.0075
0.87           0.6259      0.6140  0.0119
0.89           0.6329      0.6113  0.0216
0.90           0.6350      0.6069  0.0281
0.91           0.6366      0.6021  0.0344
0.92           0.6361      0.5963  0.0398
0.93           0.6329      0.5876  0.0454
0.94           0.6248      0.5766  0.0482
0.95           0.6125      0.5640  0.0485
0.96           0.5945      0.5523  0.0422
0.97           0.5701      0.5399  0.0302


=> 0.91 is the best threshold